In [ ]:
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, Dataset
import nltk
import re
import numpy as np
import pandas as pd
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score
from gensim.models import Word2Vec
from sklearn.feature_extraction import TfidfVectorizer
from nltk.tokenize import word_tokenize

In [ ]:
nltk.download("punkt")

In [ ]:
data = pd.read_csv("./Datasets/twitter_data.csv")
data = data.dropna()
data['category'] = data['category'].astype(int)

In [11]:
len(data)

162969

In [12]:
def preprocess(tweet):
    assert type(tweet)==str, "Invalid type provided"

    # tweet = tweet.lower()
    tweet = re.sub(r'[^a-zA-Z\s!?]', '', tweet) #removes non-alphabet characters, keeps punctuations like ! and ?
    tweet = re.sub(r'http\S+|www|S+|https\S+', '', tweet, flags=re.MULTILINE) #removes links and URLs
    tweet = re.sub(r'@\w+', '', tweet) #removes @ mentions
    tweet = re.sub(r'#', '', tweet)#removes hashtags

    tokenized_tweet = word_tokenize(tweet) #Tokenizes the tweet, separating tweet into individual tokens and joining them to form a single long string of characters

    return tokenized_tweet

In [ ]:
tweets = data['clean_text']
tokenized_tweets = [preprocess(tweet) for tweet in tweets]

In [ ]:
w2v_model = Word2Vec(vector_size=100,min_count=3,window=5,sg=1,workers=4)
w2v_model.build_vocab(tokenized_tweets)

In [ ]:
w2v_model.train(
    tokenized_tweets,
    total_examples = w2v_model.corpus_count,
    epochs=20,
    compute_loss = True,
)

vectorizer = TfidfVectorizer(max_features=1000)
X = vectorizer.fit_transform(tokenized_tweets.toarray())
y = data['']

In [ ]:
#This vectorization can definitely be improved, using something like TF-IDF vectorization
def tweet_to_vector(tweet, model=w2v_model, vector_size=100):
    tweet_vec = np.zeros(vector_size)
    count = 0
    for word in tweet:
        if word in w2v_model.wv:
            tweet_vec += w2v_model.wv[word]
            count += 1
    if count != 0:
        tweet_vec /= count
    return tweet_vec

In [ ]:
X = torch.FloatTensor([np.array(tweet_to_vector(tweet)) for tweet in tokenized_tweets])
y = torch.LongTensor(data['category'])

In [ ]:
#Craeting Pytorch datasets(and dataloaders)
class SentimentDataset(Dataset):
    def __init__(self,x,y):
        self.features = x #Represents the 100-elem vector generated by Word2Vec model, shape = [num_of_words, 100] as a Pytorch tensor
        self.labels = y - y.min() #Ensures that the targets start from 0 to num_classes-1, to make sure there is no error from CELoss saying targets out of bound

    def __len__(self):
        return len(self.features)

    def __getitem__(self,idx):
        return torch.as_tensor(self.features[idx], dtype=torch.float32), torch.as_tensor(self.labels[idx], dtype=torch.int64)

In [ ]:
x_train, x_test, y_train, y_test = train_test_split(X,y,test_size=0.2,random_state=42)

In [ ]:
train_data = SentimentDataset(x_train, y_train)
test_data = SentimentDataset(x_test, y_test)

In [ ]:
BATCH_SIZE = 32
train_dataloader = DataLoader(train_data,batch_size=BATCH_SIZE,shuffle=True,drop_last=True)
test_dataloader = DataLoader(test_data,batch_size=BATCH_SIZE,shuffle=True,drop_last=True)

In [ ]:
device = "cpu" if torch.cuda.is_available() else "cpu"

In [ ]:
class SA(nn.Module):
    def __init__(self,input_size,hidden_units,output_size):
        super().__init__()
        self.analyzer = nn.Sequential(
            nn.Linear(input_size, hidden_units),
            nn.LeakyReLU(),
            nn.Dropout(0.2),
            nn.Linear(hidden_units, hidden_units),
            nn.LeakyReLU(),
            nn.Dropout(0.2),
            nn.Linear(hidden_units, output_size),
        )       

    def forward(self, x):
        return self.analyzer(x)

Have to implement some way to map model outputs the the intended sentiment outputs - Done

In [ ]:
"""
12.11 - Upon running the model, gives an error saying that the target is out of bounds. This is because CELoss expects the targets to start from 0 to num_classes - 1. Solution for this is to create a mapping, as simple as adding 1 to the target to make them start from 1
"""

In [ ]:
INPUT_SIZE = 100
HIDDEN_UNITS = 64
OUTPUT_SIZE = 3

In [ ]:
model = SA(INPUT_SIZE, HIDDEN_UNITS, OUTPUT_SIZE).to(device)

In [ ]:
loss_fn = nn.CrossEntropyLoss()
optimizer = torch.optim.SGD(model.parameters(), lr=1e-2,weight_decay=1e-5)
epochs = 10

Introduce early stopping

In [ ]:
for epoch in range(epochs):
    train_loss, train_acc = 0,0
    for (idx, (x_train, y_train)) in enumerate(train_dataloader):
        x_train, y_train = x_train.to(device), y_train.to(device)
        # y_train = y_train + 1

        train_logits = model(x_train)
        train_preds = train_logits.argmax(dim=1)
        loss = loss_fn(train_logits, y_train).to(device)
        train_loss += loss.item()
        acc = accuracy_score(y_train, train_preds.cpu())
        train_acc += acc    
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

    train_loss /= len(train_dataloader)
    train_acc /= len(train_dataloader)
    print(f"Epoch : {epoch+1}/{epochs} | Train LOss over {len(train_dataloader)} samples : {train_loss} | Train acc over {len(train_dataloader)} samples : {train_acc:.2f}%")